# Payee extraction — LoRA fine-tuning & research plots

End-to-end notebook: **train** LoRA on payee extraction, **evaluate**, and produce **research figures** (PNG + EPS). All code is **inside this notebook** — upload only this `.ipynb` to Colab.

| Section | What you get |
|---------|----------------|
| 6–7 | Metric definitions + inline plotting library |
| 8–9 | QLoRA training + **training curves** |
| 10–11 | Full validation run + **ROC, PR, MSE, calibration**, etc. |
| 12 | Re-plot without retraining |
| 15–17 | **DP-SLM research**: theory, DP-SGD training, DP evaluation |
| 18 | Multi-ε experiment suite |


## 1. Install dependencies


In [ ]:
%pip install -q transformers datasets accelerate peft trl bitsandbytes scikit-learn tqdm matplotlib huggingface_hub opacus


## 2. GPU check

**Colab:** Runtime → Change runtime type → **GPU**.


In [ ]:
import sys
import torch
print("Python:", sys.version)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


## 3. Configuration

| Variable | Role |
|----------|------|
| `BASE_DIR` | Root with `data/train.jsonl`, `data/val.jsonl` |
| `SHOW_PLOTS` | `plt.show()` after each saved figure |
| `CALLBACK_EVAL_SAMPLES` | Val samples for **accuracy during training** (0 = off) |


In [ ]:
from pathlib import Path

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/Credit_Mitra_IITB/finetune_standalone")
else:
    BASE_DIR = Path("/content/finetune_standalone")

TRAIN_FILE = BASE_DIR / "data" / "train.jsonl"
VAL_FILE = BASE_DIR / "data" / "val.jsonl"
OUTPUT_DIR = BASE_DIR / "outputs" / "payee-lora"
EVAL_DIR = BASE_DIR / "outputs" / "eval"
PLOTS_DIR = BASE_DIR / "outputs" / "plots"
PLOTS_TRAINING_DIR = PLOTS_DIR / "training"
PLOTS_EVAL_DIR = PLOTS_DIR / "evaluation"

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRADIENT_ACCUM_STEPS = 8
LOGGING_STEPS = 20
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 2
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
MAX_NEW_TOKENS = 32
MAX_EVAL_SAMPLES = 0
SHOW_PLOTS = True
CALLBACK_EVAL_SAMPLES = 48

# ── Differential Privacy (DP-SGD) ──
USE_DP = True                    # Toggle DP training on/off
TARGET_EPSILON = 8.0             # Privacy budget (lower = more private)
TARGET_DELTA = None              # Auto-set to 1/len(train) if None
MAX_GRAD_NORM_DP = 1.0           # Per-sample gradient clipping norm
MAX_PHYSICAL_BATCH_SIZE = 2      # For Opacus BatchMemoryManager
DP_OUTPUT_DIR = BASE_DIR / "outputs" / "payee-lora-dp"
DP_EVAL_DIR = BASE_DIR / "outputs" / "eval-dp"
PLOTS_DP_DIR = PLOTS_DIR / "dp"

for p in (OUTPUT_DIR, EVAL_DIR, PLOTS_DIR, PLOTS_TRAINING_DIR, PLOTS_EVAL_DIR,
          DP_OUTPUT_DIR, DP_EVAL_DIR, PLOTS_DP_DIR):
    p.mkdir(parents=True, exist_ok=True)
print("TRAIN exists:", TRAIN_FILE.exists(), "| VAL exists:", VAL_FILE.exists())


## 4. Data format

JSONL with `prompt` (instruction + narration + `Payee:`) and `response` (gold payee). Optional `type` (e.g. P2P) for breakdown plots.


## 5. Hugging Face login (optional)


In [ ]:
# from huggingface_hub import login
# login()
pass


## 6. Shared helpers — prompts & metrics

**Theory:** Generative extraction — the model completes text after `Payee:`.

- **Exact match (EM):** `pred == gold`
- **Normalized EM (NEM):** compare after lowercase + punctuation strip + whitespace normalize
- **Char similarity:** `SequenceMatcher` ratio ∈ [0,1] — soft score for ROC/PR
- **Token Jaccard:** word overlap / union
- **MSE proxy:** `(1 − similarity)²` averaged over samples (not embedding MSE; useful for error magnitude on [0,1])


In [ ]:
import json
import re
from dataclasses import asdict, dataclass
from difflib import SequenceMatcher
from statistics import mean


def build_prompt(narration: str) -> str:
    return (
        "You are an information extraction model. Extract only the payee name "
        "from the transaction narration. Return only the payee text, with no "
        "extra words.\n\n"
        f"Transaction narration:\n{narration}\n\n"
        "Payee:"
    )


def normalize_text(text: str) -> str:
    text = (text or "").strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[\"'`.,;:!?()\[\]{}]", "", text)
    return text


def jaccard_token_similarity(a: str, b: str) -> float:
    a_set, b_set = set(normalize_text(a).split()), set(normalize_text(b).split())
    if not a_set and not b_set:
        return 1.0
    if not a_set or not b_set:
        return 0.0
    return len(a_set & b_set) / len(a_set | b_set)


def char_similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()


def extract_narration_from_prompt(prompt_field: str) -> str:
    if "Transaction narration:\n" in prompt_field and "\n\nPayee:" in prompt_field:
        return prompt_field.split("Transaction narration:\n", 1)[1].split("\n\nPayee:", 1)[0]
    return prompt_field


@dataclass
class EvalRow:
    id: str
    narration: str
    gold: str
    pred: str
    exact_match: int
    normalized_exact_match: int
    char_similarity: float
    token_jaccard: float
    mse_char: float
    mse_jaccard: float
    txn_type: str


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

print("Helpers OK.")


## 7. Plotting library (all code inline)

**Theory — training curves**

| File stem | Meaning |
|-----------|---------|
| `01_training_loss` | Cross-entropy on train batch (should decrease) |
| `02_eval_loss` | CE on validation — **detect overfitting** if train↓ but eval↑ |
| `03_learning_rate` | Scheduler / constant LR |
| `05_train_vs_eval_loss` | Overlay: gap suggests overfitting |
| `06_smoothed_train_loss` | Moving average reduces step noise |
| `07_midtraining_accuracy` | EM / NEM on a **small val subset** each eval step |
| `08_training_dashboard` | Summary panel |

**Theory — evaluation curves** (after full val inference)

| File stem | Meaning |
|-----------|---------|
| `10_accuracy_metrics_bar` | EM, NEM, mean similarities |
| `11_mse_metrics_bar` | MSE/RMSE on (1−sim)² |
| `12–13` ROC | NEM as label, similarity as **ranking score**; AUC ≈ separability |
| `14–15` PR | Precision–recall (imbalanced correct/incorrect) |
| `16–17` histograms | Distribution of similarity / per-sample MSE |
| `18–19` threshold curves | Operating point if you threshold similarity |
| `20` scatter | Char sim vs Jaccard, colored by correct/wrong |
| `21` length scatter | Over/under-generation vs gold length |
| `22` calibration | Does higher similarity bin → higher accuracy? |
| `23–24` by `type` | P2P vs other txn types |
| `25` confusion | Top-12 gold payees |


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from pathlib import Path
from statistics import mean, pstdev

from sklearn.metrics import auc, precision_recall_curve, roc_curve
from transformers import TrainerCallback
import torch
import random

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

class PayeeAccuracyCallback(TrainerCallback):
    """On each evaluation, greedy-decode payees on a small val subset and log accuracy."""

    def __init__(self, tokenizer, val_rows, max_samples=48, seed=42):
        self.tokenizer = tokenizer
        self.history = []
        rng = random.Random(seed)
        pool = list(val_rows)
        rng.shuffle(pool)
        self._subset = pool[: min(max_samples, len(pool))]

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None or not self._subset:
            return control
        model.eval()
        exact = nem = n = 0
        for row in self._subset:
            prompt_field = str(row.get("prompt", ""))
            narration = extract_narration_from_prompt(prompt_field)
            gold = str(row.get("response", "")).strip()
            prompt = build_prompt(narration)
            inputs = self.tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            pred = self.tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt) :].strip()
            exact += int(pred == gold)
            nem += int(normalize_text(pred) == normalize_text(gold))
            n += 1
        acc, nacc = exact / max(1, n), nem / max(1, n)
        self.history.append({
            "step": state.global_step,
            "accuracy": acc,
            "normalized_accuracy": nacc,
            "samples": n,
        })
        print(f"  [Callback] step={state.global_step} exact={acc:.4f} norm={nacc:.4f} n={n}")
        return control

def save_and_show(fig, plots_dir: Path, stem: str, show: bool = True) -> dict[str, str]:
    """Save figure as PNG + EPS; optionally display."""
    plots_dir = Path(plots_dir)
    plots_dir.mkdir(parents=True, exist_ok=True)
    paths = {}
    for ext in ("png", "eps"):
        path = plots_dir / f"{stem}.{ext}"
        fig.savefig(path, bbox_inches="tight", format=ext)
        paths[ext] = str(path)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return paths


def _extract_log_series(log_history: list[dict]) -> dict[str, list]:
    """Parse Hugging Face Trainer log_history into plottable series."""
    out = {
        "train_steps": [],
        "train_loss": [],
        "eval_steps": [],
        "eval_loss": [],
        "lr_steps": [],
        "lr": [],
        "grad_steps": [],
        "grad_norm": [],
        "epoch_steps": [],
        "epoch": [],
    }
    for entry in log_history:
        step = entry.get("step")
        if step is None:
            continue
        if "loss" in entry and "eval_loss" not in entry:
            out["train_steps"].append(step)
            out["train_loss"].append(entry["loss"])
        if "eval_loss" in entry:
            out["eval_steps"].append(step)
            out["eval_loss"].append(entry["eval_loss"])
        if "learning_rate" in entry:
            out["lr_steps"].append(step)
            out["lr"].append(entry["learning_rate"])
        if "grad_norm" in entry:
            out["grad_steps"].append(step)
            out["grad_norm"].append(entry["grad_norm"])
        if "epoch" in entry:
            out["epoch_steps"].append(step)
            out["epoch"].append(entry["epoch"])
    return out


def plot_training_phase(
    log_history: list[dict],
    plots_dir: Path,
    model_name: str = "",
    eval_accuracy_history: list[dict] | None = None,
    show: bool = True,
) -> list[str]:
    """
    Plots from Trainer log_history (and optional mid-training accuracy snapshots).
    Returns list of saved file stems.
    """
    plots_dir = Path(plots_dir)
    series = _extract_log_series(log_history)
    saved: list[str] = []

    def line_plot(x, y, title, ylabel, stem, color, log_y=False):
        if not x:
            print(f"Skip (no data): {title}")
            return
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(x, y, color=color, linewidth=1.5, marker="o", markersize=3)
        ax.set_xlabel("Step")
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight="bold")
        if log_y and min(y) > 0:
            ax.set_yscale("log")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_and_show(fig, plots_dir, stem, show=show)
        saved.append(stem)

    line_plot(
        series["train_steps"],
        series["train_loss"],
        f"Training loss — {model_name}".strip(" —"),
        "Cross-entropy loss",
        "01_training_loss",
        "#E74C3C",
    )
    line_plot(
        series["eval_steps"],
        series["eval_loss"],
        f"Validation loss — {model_name}".strip(" —"),
        "Eval cross-entropy loss",
        "02_eval_loss",
        "#2ECC71",
    )
    line_plot(
        series["lr_steps"],
        series["lr"],
        "Learning rate schedule",
        "Learning rate",
        "03_learning_rate",
        "#9B59B6",
    )

    if series["grad_steps"]:
        line_plot(
            series["grad_steps"],
            series["grad_norm"],
            "Gradient norm",
            "grad_norm",
            "04_gradient_norm",
            "#3498DB",
        )

    # Train vs eval loss on shared axis
    if series["train_steps"] and series["eval_steps"]:
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(
            series["train_steps"],
            series["train_loss"],
            label="Train loss",
            color="#E74C3C",
            linewidth=1.5,
        )
        ax.plot(
            series["eval_steps"],
            series["eval_loss"],
            label="Eval loss",
            color="#2ECC71",
            linewidth=1.5,
            marker="s",
            markersize=4,
        )
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title("Train vs validation loss", fontweight="bold")
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_and_show(fig, plots_dir, "05_train_vs_eval_loss", show=show)
        saved.append("05_train_vs_eval_loss")

    # Smoothed train loss (moving average)
    if len(series["train_loss"]) >= 5:
        w = min(9, len(series["train_loss"]) // 2 * 2 + 1)
        kernel = np.ones(w) / w
        smooth = np.convolve(series["train_loss"], kernel, mode="valid")
        xs = series["train_steps"][w // 2 : w // 2 + len(smooth)]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(series["train_steps"], series["train_loss"], alpha=0.35, label="Raw", color="#E74C3C")
        ax.plot(xs, smooth, label=f"MA({w})", color="#C0392B", linewidth=2)
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title("Smoothed training loss", fontweight="bold")
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_and_show(fig, plots_dir, "06_smoothed_train_loss", show=show)
        saved.append("06_smoothed_train_loss")

    # Mid-training accuracy from callback
    if eval_accuracy_history:
        steps = [h["step"] for h in eval_accuracy_history]
        acc = [h["accuracy"] for h in eval_accuracy_history]
        nem = [h.get("normalized_accuracy", h["accuracy"]) for h in eval_accuracy_history]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(steps, acc, marker="o", label="Exact match", color="#2980B9")
        ax.plot(steps, nem, marker="s", label="Normalized exact match", color="#16A085")
        ax.set_xlabel("Step")
        ax.set_ylabel("Accuracy")
        ax.set_ylim(0, 1.05)
        ax.set_title("Validation accuracy during fine-tuning", fontweight="bold")
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_and_show(fig, plots_dir, "07_midtraining_accuracy", show=show)
        saved.append("07_midtraining_accuracy")

    # Combined dashboard
    n_panels = 3 + (1 if eval_accuracy_history else 0)
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4))
    if n_panels == 1:
        axes = [axes]
    idx = 0
    if series["train_steps"]:
        axes[idx].plot(series["train_steps"], series["train_loss"], color="#E74C3C")
        axes[idx].set_title("Train loss", fontweight="bold")
        axes[idx].grid(True, alpha=0.3)
    idx += 1
    if series["eval_steps"]:
        axes[idx].plot(series["eval_steps"], series["eval_loss"], color="#2ECC71")
        axes[idx].set_title("Eval loss", fontweight="bold")
        axes[idx].grid(True, alpha=0.3)
    idx += 1
    if series["lr_steps"]:
        axes[idx].plot(series["lr_steps"], series["lr"], color="#9B59B6")
        axes[idx].set_title("Learning rate", fontweight="bold")
        axes[idx].grid(True, alpha=0.3)
    idx += 1
    if eval_accuracy_history and idx < len(axes):
        axes[idx].plot(steps, nem, color="#16A085", marker="o")
        axes[idx].set_ylim(0, 1.05)
        axes[idx].set_title("Val accuracy", fontweight="bold")
        axes[idx].grid(True, alpha=0.3)
    fig.suptitle(f"Training dashboard — {model_name}".strip(" —"), fontweight="bold", y=1.02)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "08_training_dashboard", show=show)
    saved.append("08_training_dashboard")

    return saved


def _eval_arrays(eval_rows: list) -> dict[str, np.ndarray]:
    """Convert EvalRow-like dicts/objects to numpy arrays."""
    def g(row, key):
        if isinstance(row, dict):
            return row[key]
        return getattr(row, key)

    return {
        "exact": np.array([g(r, "exact_match") for r in eval_rows], dtype=float),
        "nem": np.array([g(r, "normalized_exact_match") for r in eval_rows], dtype=float),
        "char_sim": np.array([g(r, "char_similarity") for r in eval_rows], dtype=float),
        "jaccard": np.array([g(r, "token_jaccard") for r in eval_rows], dtype=float),
        "mse_char": np.array([g(r, "mse_char") for r in eval_rows], dtype=float),
        "mse_jaccard": np.array([g(r, "mse_jaccard") for r in eval_rows], dtype=float),
        "gold_len": np.array([len(g(r, "gold") or "") for r in eval_rows]),
        "pred_len": np.array([len(g(r, "pred") or "") for r in eval_rows]),
        "types": [g(r, "txn_type") if g(r, "txn_type") else "unknown" for r in eval_rows],
    }


def plot_evaluation_phase(
    eval_rows: list,
    metrics: dict,
    plots_dir: Path,
    model_name: str = "",
    show: bool = True,
) -> list[str]:
    """Research plots from full validation predictions."""
    if roc_curve is None:
        raise ImportError("scikit-learn is required: pip install scikit-learn")

    plots_dir = Path(plots_dir)
    arr = _eval_arrays(eval_rows)
    saved: list[str] = []
    n = len(eval_rows)

    # --- Accuracy bar chart ---
    fig, ax = plt.subplots(figsize=(7, 4))
    names = ["Exact match", "Normalized\nexact match", "Mean char\nsimilarity", "Mean token\nJaccard"]
    vals = [
        metrics.get("exact_match", arr["exact"].mean()),
        metrics.get("normalized_exact_match", arr["nem"].mean()),
        metrics.get("avg_char_similarity", arr["char_sim"].mean()),
        metrics.get("avg_token_jaccard", arr["jaccard"].mean()),
    ]
    colors = ["#3498DB", "#2ECC71", "#9B59B6", "#E67E22"]
    bars = ax.bar(names, vals, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(f"Validation metrics (n={n})", fontweight="bold")
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "10_accuracy_metrics_bar", show=show)
    saved.append("10_accuracy_metrics_bar")

    # --- MSE summary ---
    fig, ax = plt.subplots(figsize=(7, 4))
    mse_labels = ["MSE (1 − char sim)", "MSE (1 − Jaccard)", "RMSE char", "RMSE Jaccard"]
    mse_vals = [
        metrics.get("mse_char", arr["mse_char"].mean()),
        metrics.get("mse_jaccard", arr["mse_jaccard"].mean()),
        metrics.get("rmse_char", np.sqrt(arr["mse_char"].mean())),
        metrics.get("rmse_jaccard", np.sqrt(arr["mse_jaccard"].mean())),
    ]
    ax.bar(mse_labels, mse_vals, color=["#E74C3C", "#C0392B", "#F39C12", "#D35400"])
    ax.set_ylabel("Error")
    ax.set_title("MSE / RMSE on similarity (research proxy)", fontweight="bold")
    fig.tight_layout()
    save_and_show(fig, plots_dir, "11_mse_metrics_bar", show=show)
    saved.append("11_mse_metrics_bar")

    # --- ROC (binary correct vs similarity score) ---
    y_true = arr["nem"]
    for score, label, stem in [
        (arr["char_sim"], "Char similarity", "12_roc_char_similarity"),
        (arr["jaccard"], "Token Jaccard", "13_roc_token_jaccard"),
    ]:
        fpr, tpr, _ = roc_curve(y_true, score)
        roc_auc = auc(fpr, tpr)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.plot(fpr, tpr, color="#2980B9", linewidth=2, label=f"AUC = {roc_auc:.4f}")
        ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
        ax.set_xlabel("False positive rate")
        ax.set_ylabel("True positive rate")
        ax.set_title(f"ROC — {label} as score", fontweight="bold")
        ax.legend(loc="lower right")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_and_show(fig, plots_dir, stem, show=show)
        saved.append(stem)

    # --- Precision-Recall ---
    for score, label, stem in [
        (arr["char_sim"], "Char similarity", "14_pr_char_similarity"),
        (arr["jaccard"], "Token Jaccard", "15_pr_token_jaccard"),
    ]:
        precision, recall, _ = precision_recall_curve(y_true, score)
        pr_auc = auc(recall, precision)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.plot(recall, precision, color="#8E44AD", linewidth=2, label=f"AUC = {pr_auc:.4f}")
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_title(f"Precision–Recall — {label}", fontweight="bold")
        ax.legend(loc="lower left")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_and_show(fig, plots_dir, stem, show=show)
        saved.append(stem)

    # --- Similarity histograms ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(arr["char_sim"], bins=30, color="#3498DB", edgecolor="white", alpha=0.85)
    axes[0].set_title("Char similarity distribution")
    axes[0].set_xlabel("Similarity")
    axes[1].hist(arr["jaccard"], bins=30, color="#2ECC71", edgecolor="white", alpha=0.85)
    axes[1].set_title("Token Jaccard distribution")
    axes[1].set_xlabel("Jaccard")
    for ax in axes:
        ax.grid(True, alpha=0.3)
    fig.suptitle("Prediction similarity to gold", fontweight="bold", y=1.02)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "16_similarity_histograms", show=show)
    saved.append("16_similarity_histograms")

    # --- MSE per-sample histogram ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(arr["mse_char"], bins=30, color="#E74C3C", edgecolor="white", alpha=0.85)
    axes[0].set_title("Per-sample MSE (char)")
    axes[1].hist(arr["mse_jaccard"], bins=30, color="#C0392B", edgecolor="white", alpha=0.85)
    axes[1].set_title("Per-sample MSE (Jaccard)")
    fig.tight_layout()
    save_and_show(fig, plots_dir, "17_mse_histograms", show=show)
    saved.append("17_mse_histograms")

    # --- Cumulative accuracy vs threshold ---
    thresholds = np.linspace(0, 1, 101)
    fig, ax = plt.subplots(figsize=(8, 4))
    for score, label, c in [
        (arr["char_sim"], "Char sim ≥ t", "#2980B9"),
        (arr["jaccard"], "Jaccard ≥ t", "#27AE60"),
    ]:
        acc_at_t = [(score >= t).mean() for t in thresholds]
        ax.plot(thresholds, acc_at_t, label=label, color=c, linewidth=1.5)
    ax.set_xlabel("Similarity threshold")
    ax.set_ylabel("Fraction of samples ≥ threshold")
    ax.set_title("Threshold vs coverage (soft accuracy)", fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "18_threshold_coverage", show=show)
    saved.append("18_threshold_coverage")

    # Strict accuracy vs threshold (pred counted correct if sim >= t AND nem)
    fig, ax = plt.subplots(figsize=(8, 4))
    for score, label, c in [
        (arr["char_sim"], "Char sim", "#8E44AD"),
        (arr["jaccard"], "Jaccard", "#16A085"),
    ]:
        strict = [((score >= t) & (arr["nem"] == 1)).mean() for t in thresholds]
        ax.plot(thresholds, strict, label=f"Correct if score≥t ({label})", linewidth=1.5)
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Strict match rate")
    ax.set_title("Threshold-based decision accuracy", fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "19_threshold_strict_accuracy", show=show)
    saved.append("19_threshold_strict_accuracy")

    # --- Scatter char vs jaccard ---
    fig, ax = plt.subplots(figsize=(7, 6))
    correct = arr["nem"] == 1
    ax.scatter(
        arr["char_sim"][~correct],
        arr["jaccard"][~correct],
        alpha=0.5,
        s=18,
        c="#E74C3C",
        label="Incorrect",
    )
    ax.scatter(
        arr["char_sim"][correct],
        arr["jaccard"][correct],
        alpha=0.5,
        s=18,
        c="#2ECC71",
        label="Correct",
    )
    ax.set_xlabel("Char similarity")
    ax.set_ylabel("Token Jaccard")
    ax.set_title("Similarity scatter (correct vs incorrect)", fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "20_similarity_scatter", show=show)
    saved.append("20_similarity_scatter")

    # --- Length scatter ---
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(arr["gold_len"], arr["pred_len"], c=arr["char_sim"], cmap="viridis", alpha=0.6, s=20)
    lim = max(arr["gold_len"].max(), arr["pred_len"].max(), 1)
    ax.plot([0, lim], [0, lim], "--", color="gray")
    ax.set_xlabel("Gold payee length (chars)")
    ax.set_ylabel("Predicted length (chars)")
    ax.set_title("Length comparison (color = char similarity)", fontweight="bold")
    cb = fig.colorbar(ax.collections[0], ax=ax)
    cb.set_label("Char sim")
    fig.tight_layout()
    save_and_show(fig, plots_dir, "21_length_scatter", show=show)
    saved.append("21_length_scatter")

    # --- Calibration (binned) ---
    n_bins = 10
    bins = np.linspace(0, 1, n_bins + 1)
    fig, ax = plt.subplots(figsize=(7, 5))
    bin_centers, bin_acc = [], []
    for i in range(n_bins):
        mask = (arr["char_sim"] >= bins[i]) & (arr["char_sim"] < bins[i + 1])
        if i == n_bins - 1:
            mask = (arr["char_sim"] >= bins[i]) & (arr["char_sim"] <= bins[i + 1])
        if mask.sum() == 0:
            continue
        bin_centers.append((bins[i] + bins[i + 1]) / 2)
        bin_acc.append(arr["nem"][mask].mean())
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
    ax.plot(bin_centers, bin_acc, "o-", color="#2980B9", label="Observed")
    ax.set_xlabel("Mean predicted score (char sim bin)")
    ax.set_ylabel("Fraction correct (NEM)")
    ax.set_title("Calibration curve (char similarity)", fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "22_calibration_char", show=show)
    saved.append("22_calibration_char")

    # --- By transaction type ---
    types = arr["types"]
    unique_types = sorted(set(types))
    if len(unique_types) > 1 and len(unique_types) <= 25:
        type_acc = []
        type_n = []
        for t in unique_types:
            m = np.array([ti == t for ti in types])
            type_acc.append(arr["nem"][m].mean())
            type_n.append(m.sum())
        fig, ax = plt.subplots(figsize=(max(8, len(unique_types) * 0.5), 4))
        xpos = np.arange(len(unique_types))
        ax.bar(xpos, type_acc, color="#3498DB", edgecolor="black", linewidth=0.4)
        ax.set_xticks(xpos)
        ax.set_xticklabels([f"{t}\n(n={n})" for t, n in zip(unique_types, type_n)], rotation=45, ha="right")
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Normalized exact match")
        ax.set_title("Accuracy by transaction type", fontweight="bold")
        fig.tight_layout()
        save_and_show(fig, plots_dir, "23_accuracy_by_type", show=show)
        saved.append("23_accuracy_by_type")

        fig, ax = plt.subplots(figsize=(max(8, len(unique_types) * 0.5), 4))
        data = [arr["char_sim"][np.array([ti == t for ti in types])] for t in unique_types]
        ax.boxplot(data, labels=unique_types)
        ax.set_ylabel("Char similarity")
        ax.set_title("Similarity by transaction type", fontweight="bold")
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
        fig.tight_layout()
        save_and_show(fig, plots_dir, "24_boxplot_similarity_by_type", show=show)
        saved.append("24_boxplot_similarity_by_type")

    # --- Top payees confusion (subset) ---
    from collections import Counter

    def _field(row, key):
        return row[key] if isinstance(row, dict) else getattr(row, key)

    golds = [_field(r, "gold") for r in eval_rows]
    preds = [_field(r, "pred") for r in eval_rows]
    gold_counts = Counter(golds)
    top_golds = [g for g, _ in gold_counts.most_common(12)]
    if top_golds:
        idx_map = {g: i for i, g in enumerate(top_golds)}
        cm = np.zeros((len(top_golds), len(top_golds)))
        for g, p in zip(golds, preds):
            if g in idx_map and p in idx_map:
                cm[idx_map[g], idx_map[p]] += 1
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(top_golds)))
        ax.set_yticks(range(len(top_golds)))
        ax.set_xticklabels(top_golds, rotation=45, ha="right", fontsize=8)
        ax.set_yticklabels(top_golds, fontsize=8)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Gold")
        ax.set_title("Confusion matrix (top 12 gold payees)", fontweight="bold")
        fig.colorbar(im, ax=ax)
        fig.tight_layout()
        save_and_show(fig, plots_dir, "25_confusion_top_payees", show=show)
        saved.append("25_confusion_top_payees")

    # Save extended metrics JSON
    research = dict(metrics)
    fpr_c, tpr_c, _ = roc_curve(y_true, arr["char_sim"])
    fpr_j, tpr_j, _ = roc_curve(y_true, arr["jaccard"])
    research["roc_auc_char"] = float(auc(fpr_c, tpr_c))
    research["roc_auc_jaccard"] = float(auc(fpr_j, tpr_j))
    p, r, _ = precision_recall_curve(y_true, arr["char_sim"])
    research["pr_auc_char"] = float(auc(r, p))
    research["mse_char_std"] = float(pstdev(arr["mse_char"].tolist())) if n > 1 else 0.0
    out_path = plots_dir.parent.parent / "eval" / "research_metrics.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(research, f, indent=2, ensure_ascii=False)
    print("Saved research metrics:", out_path)

    return saved


# ═══════════════════════════════════════════════════════════════════════════
# Differential Privacy (DP-SGD) Plots
# ═══════════════════════════════════════════════════════════════════════════


def plot_dp_epsilon_convergence(
    dp_history: list[dict],
    plots_dir: Path,
    target_epsilon: float | None = None,
    show: bool = True,
) -> str:
    """Plot cumulative epsilon consumed over training steps.

    dp_history: list of dicts with keys 'step', 'epsilon', 'loss' (logged each step).
    """
    plots_dir = Path(plots_dir)
    steps = [h["step"] for h in dp_history]
    epsilons = [h["epsilon"] for h in dp_history]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(steps, epsilons, color="#E74C3C", linewidth=2, label="ε spent")
    if target_epsilon is not None and target_epsilon < float("inf"):
        ax.axhline(y=target_epsilon, color="#95A5A6", linestyle="--", linewidth=1.5,
                   label=f"Target ε = {target_epsilon}")
    ax.set_xlabel("Training step")
    ax.set_ylabel("Cumulative ε (privacy budget spent)")
    ax.set_title("Privacy budget consumption during training", fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "30_dp_epsilon_convergence", show=show)
    return "30_dp_epsilon_convergence"


def plot_dp_privacy_utility_tradeoff(
    experiment_results: list[dict],
    plots_dir: Path,
    show: bool = True,
) -> str:
    """Bar chart comparing accuracy metrics across different epsilon values.

    experiment_results: list of dicts, each with keys:
        'label', 'epsilon', 'exact_match', 'normalized_exact_match',
        'avg_char_similarity', 'avg_token_jaccard'
    """
    plots_dir = Path(plots_dir)
    n_exp = len(experiment_results)
    labels = [r["label"] for r in experiment_results]
    metric_names = ["Exact Match", "Normalized EM", "Char Similarity", "Token Jaccard"]
    metric_keys = ["exact_match", "normalized_exact_match", "avg_char_similarity", "avg_token_jaccard"]
    colors = ["#3498DB", "#2ECC71", "#9B59B6", "#E67E22"]

    x = np.arange(n_exp)
    width = 0.18
    fig, ax = plt.subplots(figsize=(max(10, n_exp * 3), 5))

    for i, (mk, mn, c) in enumerate(zip(metric_keys, metric_names, colors)):
        vals = [r.get(mk, 0) for r in experiment_results]
        offset = (i - len(metric_names) / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=mn, color=c, edgecolor="black", linewidth=0.4)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title("Privacy–Utility Tradeoff: Accuracy vs Privacy Budget (ε)", fontweight="bold")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, alpha=0.3, axis="y")
    fig.tight_layout()
    save_and_show(fig, plots_dir, "31_dp_privacy_utility_tradeoff", show=show)
    return "31_dp_privacy_utility_tradeoff"


def plot_dp_vs_nondp_comparison(
    nondp_metrics: dict,
    dp_metrics: dict,
    plots_dir: Path,
    dp_epsilon: float = 8.0,
    show: bool = True,
) -> str:
    """Side-by-side comparison of DP vs non-DP metrics."""
    plots_dir = Path(plots_dir)
    metric_names = ["Exact Match", "Normalized EM", "Char Similarity", "Token Jaccard"]
    metric_keys = ["exact_match", "normalized_exact_match", "avg_char_similarity", "avg_token_jaccard"]

    nondp_vals = [nondp_metrics.get(k, 0) for k in metric_keys]
    dp_vals = [dp_metrics.get(k, 0) for k in metric_keys]

    x = np.arange(len(metric_names))
    width = 0.35
    fig, ax = plt.subplots(figsize=(9, 5))
    bars1 = ax.bar(x - width / 2, nondp_vals, width, label="Non-DP (ε = ∞)",
                   color="#2ECC71", edgecolor="black", linewidth=0.4)
    bars2 = ax.bar(x + width / 2, dp_vals, width, label=f"DP-SGD (ε = {dp_epsilon})",
                   color="#E74C3C", edgecolor="black", linewidth=0.4)

    for bars in (bars1, bars2):
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f"{h:.3f}",
                    ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title("DP vs Non-DP: Payee Extraction Accuracy", fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    fig.tight_layout()
    save_and_show(fig, plots_dir, "32_dp_vs_nondp_comparison", show=show)
    return "32_dp_vs_nondp_comparison"


def plot_dp_gradient_norm_distribution(
    grad_norms: list[float],
    max_grad_norm: float,
    plots_dir: Path,
    show: bool = True,
) -> str:
    """Histogram of per-sample gradient norms, with clipping threshold line."""
    plots_dir = Path(plots_dir)
    fig, ax = plt.subplots(figsize=(8, 4))

    norms = np.array(grad_norms)
    ax.hist(norms, bins=50, color="#3498DB", edgecolor="white", alpha=0.85, density=True)
    ax.axvline(x=max_grad_norm, color="#E74C3C", linestyle="--", linewidth=2,
               label=f"Clipping threshold C = {max_grad_norm}")

    clipped_frac = (norms > max_grad_norm).mean() * 100
    ax.set_xlabel("Per-sample gradient norm")
    ax.set_ylabel("Density")
    ax.set_title(f"Gradient norm distribution ({clipped_frac:.1f}% clipped)", fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "33_dp_gradient_norm_distribution", show=show)
    return "33_dp_gradient_norm_distribution"


def plot_dp_training_dashboard(
    dp_history: list[dict],
    plots_dir: Path,
    target_epsilon: float | None = None,
    model_name: str = "",
    show: bool = True,
) -> str:
    """Combined DP training dashboard: loss, epsilon, and noise over steps."""
    plots_dir = Path(plots_dir)
    steps = [h["step"] for h in dp_history]
    losses = [h.get("loss", 0) for h in dp_history]
    epsilons = [h.get("epsilon", 0) for h in dp_history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Loss
    axes[0].plot(steps, losses, color="#E74C3C", linewidth=1.5)
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("DP Training Loss", fontweight="bold")
    axes[0].grid(True, alpha=0.3)

    # Epsilon
    axes[1].plot(steps, epsilons, color="#2980B9", linewidth=1.5)
    if target_epsilon is not None and target_epsilon < float("inf"):
        axes[1].axhline(y=target_epsilon, color="#95A5A6", linestyle="--",
                       label=f"Target ε = {target_epsilon}")
        axes[1].legend()
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("ε (cumulative)")
    axes[1].set_title("Privacy Budget Consumption", fontweight="bold")
    axes[1].grid(True, alpha=0.3)

    title = f"DP-SGD Training Dashboard — {model_name}".strip(" —")
    fig.suptitle(title, fontweight="bold", y=1.02)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "34_dp_training_dashboard", show=show)
    return "34_dp_training_dashboard"


def plot_dp_loss_comparison(
    nondp_log_history: list[dict],
    dp_history: list[dict],
    plots_dir: Path,
    dp_epsilon: float = 8.0,
    show: bool = True,
) -> str:
    """Overlay training loss curves for DP vs non-DP runs."""
    plots_dir = Path(plots_dir)
    fig, ax = plt.subplots(figsize=(9, 4))

    # Non-DP losses
    nondp_series = _extract_log_series(nondp_log_history)
    if nondp_series["train_steps"]:
        ax.plot(nondp_series["train_steps"], nondp_series["train_loss"],
                color="#2ECC71", linewidth=1.5, alpha=0.8, label="Non-DP (ε = ∞)")

    # DP losses
    dp_steps = [h["step"] for h in dp_history]
    dp_losses = [h.get("loss", 0) for h in dp_history]
    if dp_steps:
        ax.plot(dp_steps, dp_losses, color="#E74C3C", linewidth=1.5, alpha=0.8,
                label=f"DP-SGD (ε = {dp_epsilon})")

    ax.set_xlabel("Step")
    ax.set_ylabel("Training Loss")
    ax.set_title("Training Loss: DP vs Non-DP", fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_and_show(fig, plots_dir, "35_dp_loss_comparison", show=show)
    return "35_dp_loss_comparison"


def plot_all(
    log_history: list[dict] | None,
    eval_rows: list | None,
    metrics: dict | None,
    plots_training_dir: Path,
    plots_eval_dir: Path,
    model_name: str = "",
    eval_accuracy_history: list[dict] | None = None,
    show: bool = True,
) -> dict[str, list[str]]:
    """Run training and/or evaluation plot suites."""
    result = {"training": [], "evaluation": []}
    if log_history:
        print("\n=== Training-phase plots ===")
        result["training"] = plot_training_phase(
            log_history,
            plots_training_dir,
            model_name=model_name,
            eval_accuracy_history=eval_accuracy_history,
            show=show,
        )
    if eval_rows and metrics:
        print("\n=== Evaluation-phase plots ===")
        result["evaluation"] = plot_evaluation_phase(
            eval_rows,
            metrics,
            plots_eval_dir,
            model_name=model_name,
            show=show,
        )
    return result

print("Inline plotting library ready.")


## 8. Fine-tuning (QLoRA + SFTTrainer)

**Theory: LoRA & 4-bit Quantization (QLoRA)**
- **LoRA (Low-Rank Adaptation):** Instead of updating all parameters of the 135M model, we freeze the original weights and inject trainable rank decomposition matrices (adapters) into the attention layers (`q_proj`, `k_proj`, etc.). This reduces trainable parameters from ~135M to ~1M, slashing memory usage.
- **4-bit Quantization (NF4):** The frozen base model is loaded in 4-bit precision instead of 16-bit or 32-bit. This drastically reduces VRAM requirements, allowing the model to fit on a standard Colab T4 GPU.
- **Effective Batch Size:** Training uses gradient accumulation. True batch size = `BATCH_SIZE × GRADIENT_ACCUM_STEPS`.

**Run this cell once** — it saves the adapter to `outputs/payee-lora/`. Then run **Section 9** for training plots.


In [ ]:
import os
import inspect
import torch
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

assert TRAIN_FILE.exists(), f"Missing: {TRAIN_FILE}"
assert VAL_FILE.exists(), f"Missing: {VAL_FILE}"

dataset = load_dataset(
    "json",
    data_files={"train": str(TRAIN_FILE), "validation": str(VAL_FILE)},
)
if "response" in dataset["train"].column_names and "completion" not in dataset["train"].column_names:
    dataset = dataset.map(lambda x: {"completion": x["response"]})

print("Train:", len(dataset["train"]), "| Val:", len(dataset["validation"]))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_cuda = torch.cuda.is_available()
print("CUDA:", use_cuda, torch.cuda.get_device_name(0) if use_cuda else "CPU")

use_quant = False
model_kwargs = {"device_map": "auto"}
if use_cuda:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    use_quant = True
else:
    model_kwargs["torch_dtype"] = torch.float32

try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
except Exception as e:
    print("Quantized load failed, fallback:", e)
    use_quant = False
    model_kwargs.pop("quantization_config", None)
    model_kwargs["torch_dtype"] = torch.float16 if use_cuda else torch.float32
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)

model.config.use_cache = False
if use_quant:
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

peft_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM", target_modules=LORA_TARGET_MODULES,
)

training_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=LOGGING_STEPS,
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    bf16=False,
    fp16=False,
    optim="adamw_torch",
    report_to="none",
)
sig = inspect.signature(TrainingArguments.__init__)
training_kwargs["evaluation_strategy" if "evaluation_strategy" in sig.parameters else "eval_strategy"] = "steps"
training_args = TrainingArguments(**training_kwargs)

val_rows_for_callback = load_jsonl(VAL_FILE)
callbacks = []
payee_callback = None
if CALLBACK_EVAL_SAMPLES > 0:
    payee_callback = PayeeAccuracyCallback(tokenizer, val_rows_for_callback, max_samples=CALLBACK_EVAL_SAMPLES)
    callbacks.append(payee_callback)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=peft_config,
    callbacks=callbacks,
)

if use_cuda:
    torch.cuda.empty_cache()
print("Starting training...")
trainer.train()
trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved LoRA to", OUTPUT_DIR)


## 9. Training curves — run after Section 8

**Theory: Cross-Entropy Loss vs Target Accuracy**
- `trainer.state.log_history` records scalars each `logging_steps` / `eval_steps`. 
- **Loss:** The metric tracked here is token-level **Cross-Entropy (CE) Loss**, measuring the model's confidence in predicting the exact next token. While CE loss correlates with our end goal (payee extraction), it is not perfectly equivalent to Exact Match (EM). A model might have high loss on the exact token (e.g. predicting "Reliance" instead of "Reliance Fresh") but still be highly useful.
- **Callback Accuracy:** If `CALLBACK_EVAL_SAMPLES > 0`, the training loop pauses periodically to run *generative inference* (greedy decoding) on a validation subset. This produces **accuracy vs step** curves that are independent of CE loss, giving a truer sense of real-world performance during training.


In [ ]:
eval_accuracy_history = payee_callback.history if payee_callback else None
training_stems = plot_training_phase(
    trainer.state.log_history,
    PLOTS_TRAINING_DIR,
    model_name=MODEL_NAME,
    eval_accuracy_history=eval_accuracy_history,
    show=SHOW_PLOTS,
)
print(f"Done: {len(training_stems)} training figures in {PLOTS_TRAINING_DIR}")


## 10. Full validation evaluation

**Theory: Inference Strategy for Extraction**
- **Greedy Decoding:** We use `do_sample=False` and `max_new_tokens=32`. For information extraction tasks (unlike creative writing or chatting), we want the single most probable sequence of tokens. This ensures deterministic, reproducible results without hallucinations.
- **Evaluation Loop:** Each validation row generates one prediction. This prediction is compared to the gold standard payee via the strict (Exact Match) and soft (Jaccard, Char Sim) metrics defined in Section 6.

Outputs: `metrics.json`, `predictions.jsonl`, `errors_top20.jsonl`. Then run **Section 11** for evaluation plots.


In [ ]:
import torch
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

val_rows = load_jsonl(VAL_FILE)
if MAX_EVAL_SAMPLES > 0:
    val_rows = val_rows[:MAX_EVAL_SAMPLES]
print("Evaluating", len(val_rows), "samples...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
model.eval()


def predict_payee(narration: str) -> str:
    prompt = build_prompt(narration)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()


eval_rows = []
for row in tqdm(val_rows, desc="Evaluating"):
    narration = extract_narration_from_prompt(str(row.get("prompt", "")))
    gold = str(row.get("response", "")).strip()
    pred = predict_payee(narration)
    cs = char_similarity(pred, gold)
    tj = jaccard_token_similarity(pred, gold)
    eval_rows.append(EvalRow(
        id=str(row.get("id", "")),
        narration=narration,
        gold=gold,
        pred=pred,
        exact_match=int(pred == gold),
        normalized_exact_match=int(normalize_text(pred) == normalize_text(gold)),
        char_similarity=cs,
        token_jaccard=tj,
        mse_char=(1.0 - cs) ** 2,
        mse_jaccard=(1.0 - tj) ** 2,
        txn_type=str(row.get("type", "unknown")),
    ))

n = len(eval_rows)
metrics = {
    "samples": n,
    "exact_match": sum(r.exact_match for r in eval_rows) / max(1, n),
    "normalized_exact_match": sum(r.normalized_exact_match for r in eval_rows) / max(1, n),
    "avg_char_similarity": mean(r.char_similarity for r in eval_rows) if n else 0.0,
    "avg_token_jaccard": mean(r.token_jaccard for r in eval_rows) if n else 0.0,
    "mse_char": mean(r.mse_char for r in eval_rows) if n else 0.0,
    "mse_jaccard": mean(r.mse_jaccard for r in eval_rows) if n else 0.0,
    "rmse_char": (mean(r.mse_char for r in eval_rows) ** 0.5) if n else 0.0,
    "rmse_jaccard": (mean(r.mse_jaccard for r in eval_rows) ** 0.5) if n else 0.0,
}

EVAL_DIR.mkdir(parents=True, exist_ok=True)
with open(EVAL_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
with open(EVAL_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
    for r in eval_rows:
        f.write(json.dumps(asdict(r), ensure_ascii=False) + "\n")
errors = sorted([r for r in eval_rows if r.normalized_exact_match == 0], key=lambda x: x.char_similarity)
with open(EVAL_DIR / "errors_top20.jsonl", "w", encoding="utf-8") as f:
    for r in errors[:20]:
        f.write(json.dumps(asdict(r), ensure_ascii=False) + "\n")

print(json.dumps(metrics, indent=2))


## 11. Evaluation curves — run after Section 10

**Theory: Advanced Evaluation Metrics**

- **ROC (Receiver Operating Characteristic):** We treat Normalized Exact Match (NEM) as the binary ground truth, and our soft similarities (Char Sim / Jaccard) as the prediction "score". The ROC curve shows the tradeoff between True Positive Rate and False Positive Rate as we vary the similarity threshold. The AUC (Area Under Curve) indicates how well the soft score separates perfect matches from incorrect ones.
- **PR (Precision-Recall):** Similar to ROC, but more informative when class balance is highly skewed (e.g., if the model easily gets 90% of payees correct, PR highlights the difficulty of the remaining 10%).
- **MSE (Mean Squared Error) Bars:** We report mean `(1−sim)²`. Think of this as the squared error on a 0–1 similarity scale (0 = perfect match, 1 = completely different). This helps quantify the magnitude of the errors when the model is wrong.

All figures saved to `outputs/plots/evaluation/` as `.png` and `.eps`.


In [ ]:
evaluation_stems = plot_evaluation_phase(
    eval_rows,
    metrics,
    PLOTS_EVAL_DIR,
    model_name=MODEL_NAME,
    show=SHOW_PLOTS,
)
print(f"Done: {len(evaluation_stems)} evaluation figures in {PLOTS_EVAL_DIR}")


## 12. Re-plot everything (no retrain)

Reload `predictions.jsonl` if needed; reuse `trainer.state.log_history` from Section 8.


In [ ]:
if "eval_rows" not in dir():
    eval_rows = []
    with open(EVAL_DIR / "predictions.jsonl", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                eval_rows.append(json.loads(line))
    with open(EVAL_DIR / "metrics.json", encoding="utf-8") as f:
        metrics = json.load(f)

hist = payee_callback.history if payee_callback else None
summary = plot_all(
    log_history=trainer.state.log_history,
    eval_rows=eval_rows,
    metrics=metrics,
    eval_accuracy_history=hist,
    show=SHOW_PLOTS,
)
summary


## 15. Research: DP-SLM (Differentially Private Small Language Models)

**Why Differential Privacy for Financial NLP?**
Bank statements contain highly sensitive Personally Identifiable Information (PII) such as payee names, account numbers, and UPI IDs. Standard fine-tuning of Large Language Models (LLMs) poses a risk: models can memorize and later leak exact training data. **Differential Privacy (DP)** provides a mathematical guarantee against this memorization.

**What is DP-SGD?**
Differentially Private Stochastic Gradient Descent modifies the standard training loop by:
1. **Clipping:** Bounding the maximum gradient norm ($C$) for each individual training sample.
2. **Noise Injection:** Adding calibrated Gaussian noise (proportional to $C$) to the aggregated batch gradient.

**The DP-SLM Paradigm**
Applying DP-SGD directly to massive LLMs (like Llama-3 70B) is computationally prohibitive and destroys model utility due to the sheer volume of noise added (noise scales with the number of parameters).
The **DP-SLM paradigm** solves this by:
- Using a **Small Language Model** (e.g., Qwen2.5-1.5B or SmolLM2-135M).
- Using **Parameter-Efficient Fine-Tuning (PEFT/LoRA)** to train only ~1% of the weights.
- Because we train far fewer parameters, the required DP noise is drastically reduced, allowing the model to learn the task while remaining strictly private.


## 16. DP-SGD Fine-tuning (Custom Loop)

**Theory:**
We cannot use HuggingFace `SFTTrainer` for DP-SGD because Opacus requires deep integration with the PyTorch `Optimizer` and `DataLoader` to compute per-sample gradients.
Furthermore, Opacus is incompatible with 4-bit quantized layers. The solution is to keep the base model frozen in 4-bit, and apply DP-SGD **only to the LoRA adapters** (which are standard FP16/32).

*This cell will train a private adapter and save it to `outputs/payee-lora-dp/`.*


In [ ]:
if USE_DP:
    
    import os
    import torch
    import warnings
    from datasets import load_dataset
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from torch.utils.data import DataLoader
    from opacus import PrivacyEngine
    from opacus.utils.batch_memory_manager import BatchMemoryManager
    from opacus.validators import ModuleValidator
    from tqdm.auto import tqdm
    
    # We must tokenize manually for the custom loop
    def tokenize_for_dp(examples, tokenizer, max_length=128):
        prompts = examples["prompt"]
        completions = examples["response"] if "response" in examples else examples["completion"]
        full_texts = [p + c for p, c in zip(prompts, completions)]
        
        tokenized = tokenizer(
            full_texts,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )
        # Causal LM: labels are the same as input_ids
        tokenized["labels"] = tokenized["input_ids"].clone()
        
        # Ignore padding tokens in loss computation
        tokenized["labels"][tokenized["input_ids"] == tokenizer.pad_token_id] = -100
        
        return tokenized
    
    print("Loading dataset for DP training...")
    dataset = load_dataset(
        "json",
        data_files={"train": str(TRAIN_FILE), "validation": str(VAL_FILE)},
    )
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Prepare training data
    train_dataset = dataset["train"].map(
        lambda x: tokenize_for_dp(x, tokenizer),
        batched=True,
        remove_columns=dataset["train"].column_names
    )
    train_dataset.set_format(type="torch")
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    print(f"Loading base model {MODEL_NAME} in 4-bit...")
    use_cuda = torch.cuda.is_available()
    
    # 4-bit Quantization (QLoRA) config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config if use_cuda else None,
        torch_dtype=torch.float16 if use_cuda else torch.float32,
        device_map="auto",
    )
    model.config.use_cache = False
    if use_cuda:
        model = prepare_model_for_kbit_training(model)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    
    print("Applying LoRA adapters...")
    peft_config = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM", target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(model, peft_config)
    
    # IMPORTANT: Ensure ONLY LoRA parameters require gradients
    for name, param in model.named_parameters():
        if "lora" not in name.lower():
            param.requires_grad = False
        else:
            # Opacus requires parameters requiring gradients to be float32 or float16, not int8/4
            # Since LoRA params are added by PEFT, they are already standard floats.
            pass
    
    # Fix any module incompatibilities (e.g. LayerNorm -> GroupNorm for DP)
    # We only do this strictly for parameters that require gradients.
    errors = ModuleValidator.validate(model, strict=False)
    if errors:
        print(f"Opacus ModuleValidator found {len(errors)} issues. Applying fixes...")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = ModuleValidator.fix(model)
    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE
    )
    
    print(f"Attaching Opacus PrivacyEngine (Target ε = {TARGET_EPSILON})...")
    privacy_engine = PrivacyEngine()
    target_delta = TARGET_DELTA or (1.0 / len(train_dataset))
    
    model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        epochs=NUM_EPOCHS,
        target_epsilon=TARGET_EPSILON,
        target_delta=target_delta,
        max_grad_norm=MAX_GRAD_NORM_DP,
    )
    
    print(f"Starting DP-SGD Training... (Memory-safe physical batch size: {MAX_PHYSICAL_BATCH_SIZE})")
    model.train()
    dp_log_history = []
    global_step = 0
    
    for epoch in range(NUM_EPOCHS):
        epoch_loss = 0
        with BatchMemoryManager(
            data_loader=train_loader, 
            max_physical_batch_size=MAX_PHYSICAL_BATCH_SIZE, 
            optimizer=optimizer
        ) as memory_safe_loader:
            
            pbar = tqdm(memory_safe_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
            for i, batch in enumerate(pbar):
                # Batch memory manager handles the virtual batching loop
                
                optimizer.zero_grad()
                outputs = model(
                    input_ids=batch["input_ids"].to(model.device),
                    attention_mask=batch["attention_mask"].to(model.device),
                    labels=batch["labels"].to(model.device)
                )
                loss = outputs.loss
                
                loss.backward()
                optimizer.step()
                
                # Opacus updates the gradients and takes optimization steps according to logical batches.
                # We track loss continuously.
                epoch_loss += loss.item()
                
                if optimizer.step_was_taken:
                    global_step += 1
                    current_epsilon = privacy_engine.get_epsilon(target_delta)
                    
                    # Log metrics
                    if global_step % LOGGING_STEPS == 0:
                        dp_log_history.append({
                            "step": global_step,
                            "epoch": epoch + (i / len(memory_safe_loader)),
                            "loss": loss.item(),
                            "epsilon": current_epsilon,
                            "grad_norm": model.grad_sample_norm() if hasattr(model, 'grad_sample_norm') else 0,
                        })
                        pbar.set_postfix({"Loss": f"{loss.item():.4f}", "ε": f"{current_epsilon:.2f}"})
    
        current_epsilon = privacy_engine.get_epsilon(target_delta)
        print(f"End of Epoch {epoch+1} — Loss: {epoch_loss/len(memory_safe_loader):.4f} — ε: {current_epsilon:.2f} (δ: {target_delta})")
    
    # Save the DP-trained adapter
    print(f"Saving DP-trained LoRA adapter to {DP_OUTPUT_DIR}...")
    # Remove the opacus wrapper from the model to save correctly using PEFT
    unwrapped_model = model._module
    unwrapped_model.save_pretrained(str(DP_OUTPUT_DIR))
    tokenizer.save_pretrained(str(DP_OUTPUT_DIR))
    print("DP Training Complete!")


## 17. DP Evaluation & Comparison

Let's evaluate the DP model and compare it against the Non-DP baseline.


In [ ]:
if USE_DP:
    
    import torch
    from peft import PeftModel
    from tqdm.auto import tqdm
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    val_rows = load_jsonl(VAL_FILE)
    if MAX_EVAL_SAMPLES > 0:
        val_rows = val_rows[:MAX_EVAL_SAMPLES]
    print("Evaluating DP Model on", len(val_rows), "samples...")
    
    tokenizer = AutoTokenizer.from_pretrained(str(DP_OUTPUT_DIR))
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(base_model, str(DP_OUTPUT_DIR))
    model.eval()
    
    def predict_payee_dp(narration: str) -> str:
        prompt = build_prompt(narration)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
    
    dp_eval_rows = []
    for row in tqdm(val_rows, desc="Evaluating DP"):
        narration = extract_narration_from_prompt(str(row.get("prompt", "")))
        gold = str(row.get("response", "")).strip()
        pred = predict_payee_dp(narration)
        cs = char_similarity(pred, gold)
        tj = jaccard_token_similarity(pred, gold)
        dp_eval_rows.append(EvalRow(
            id=str(row.get("id", "")),
            narration=narration,
            gold=gold,
            pred=pred,
            exact_match=int(pred == gold),
            normalized_exact_match=int(normalize_text(pred) == normalize_text(gold)),
            char_similarity=cs,
            token_jaccard=tj,
            mse_char=(1.0 - cs) ** 2,
            mse_jaccard=(1.0 - tj) ** 2,
            txn_type=str(row.get("type", "unknown")),
        ))
    
    n = len(dp_eval_rows)
    dp_metrics = {
        "samples": n,
        "exact_match": sum(r.exact_match for r in dp_eval_rows) / max(1, n),
        "normalized_exact_match": sum(r.normalized_exact_match for r in dp_eval_rows) / max(1, n),
        "avg_char_similarity": mean(r.char_similarity for r in dp_eval_rows) if n else 0.0,
        "avg_token_jaccard": mean(r.token_jaccard for r in dp_eval_rows) if n else 0.0,
        "mse_char": mean(r.mse_char for r in dp_eval_rows) if n else 0.0,
        "mse_jaccard": mean(r.mse_jaccard for r in dp_eval_rows) if n else 0.0,
        "rmse_char": (mean(r.mse_char for r in dp_eval_rows) ** 0.5) if n else 0.0,
        "rmse_jaccard": (mean(r.mse_jaccard for r in dp_eval_rows) ** 0.5) if n else 0.0,
    }
    
    DP_EVAL_DIR.mkdir(parents=True, exist_ok=True)
    with open(DP_EVAL_DIR / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(dp_metrics, f, indent=2, ensure_ascii=False)
    with open(DP_EVAL_DIR / "predictions.jsonl", "w", encoding="utf-8") as f:
        for r in dp_eval_rows:
            f.write(json.dumps(asdict(r), ensure_ascii=False) + "\n")
    
    print("DP Metrics:")
    print(json.dumps(dp_metrics, indent=2))


In [ ]:
if USE_DP:
    print("--- DP Training Dashboard ---")
    plot_dp_training_dashboard(dp_log_history, PLOTS_DP_DIR, target_epsilon=TARGET_EPSILON, model_name=MODEL_NAME, show=SHOW_PLOTS)
    
    print("--- DP vs Non-DP Comparison ---")
    plot_dp_vs_nondp_comparison(metrics, dp_metrics, PLOTS_DP_DIR, dp_epsilon=TARGET_EPSILON, show=SHOW_PLOTS)
    
    if len(dp_log_history) > 0 and 'grad_norm' in dp_log_history[0]:
        plot_dp_gradient_norm_distribution([h['grad_norm'] for h in dp_log_history], MAX_GRAD_NORM_DP, PLOTS_DP_DIR, show=SHOW_PLOTS)


## 18. Multi-Epsilon Experiment Suite (Optional)

Run the DP training loop multiple times across different privacy budgets to plot the **Privacy-Utility Tradeoff**.
(Uncomment and run if you have time/compute).


In [ ]:
import copy

EXPERIMENTS = [
    {"epsilon": float("inf"), "label": "Non-DP (baseline)"},
    {"epsilon": 8.0, "label": "DP (ε=8, relaxed)"},
    {"epsilon": 3.0, "label": "DP (ε=3, moderate)"},
    {"epsilon": 1.0, "label": "DP (ε=1, strict)"},
]
results = []

# For the baseline, we already have the metrics from Section 10
if "metrics" in globals():
    baseline_result = dict(metrics)
    baseline_result["label"] = "Non-DP (baseline)"
    baseline_result["epsilon"] = float("inf")
    results.append(baseline_result)

# We will define a helper to run DP for a specific epsilon
def run_dp_experiment(epsilon, label):
    print(f"\n{'='*50}\nRunning DP Experiment: {label}\n{'='*50}")
    
    # 1. Re-initialize model and optimizer
    exp_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config if use_cuda else None,
        torch_dtype=torch.float16 if use_cuda else torch.float32,
        device_map="auto",
    )
    exp_model.config.use_cache = False
    if use_cuda:
        exp_model = prepare_model_for_kbit_training(exp_model)
        exp_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        
    exp_model = get_peft_model(exp_model, peft_config)
    
    for name, param in exp_model.named_parameters():
        if "lora" not in name.lower():
            param.requires_grad = False
            
    _ = ModuleValidator.fix(exp_model)
    
    exp_optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, exp_model.parameters()),
        lr=LEARNING_RATE
    )
    
    # 2. Attach PrivacyEngine
    exp_privacy_engine = PrivacyEngine()
    exp_target_delta = TARGET_DELTA or (1.0 / len(train_dataset))
    
    exp_model, exp_optimizer, exp_train_loader = exp_privacy_engine.make_private_with_epsilon(
        module=exp_model,
        optimizer=exp_optimizer,
        data_loader=train_loader,
        epochs=NUM_EPOCHS,
        target_epsilon=epsilon,
        target_delta=exp_target_delta,
        max_grad_norm=MAX_GRAD_NORM_DP,
    )
    
    # 3. Train
    exp_model.train()
    for epoch in range(NUM_EPOCHS):
        with BatchMemoryManager(
            data_loader=exp_train_loader, 
            max_physical_batch_size=MAX_PHYSICAL_BATCH_SIZE, 
            optimizer=exp_optimizer
        ) as memory_safe_loader:
            for batch in tqdm(memory_safe_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} (ε={epsilon})"):
                exp_optimizer.zero_grad()
                outputs = exp_model(
                    input_ids=batch["input_ids"].to(exp_model.device),
                    attention_mask=batch["attention_mask"].to(exp_model.device),
                    labels=batch["labels"].to(exp_model.device)
                )
                loss = outputs.loss
                loss.backward()
                exp_optimizer.step()
                
    # 4. Evaluate
    exp_model.eval()
    exp_eval_rows = []
    
    def exp_predict(narration: str) -> str:
        prompt = build_prompt(narration)
        inputs = tokenizer(prompt, return_tensors="pt").to(exp_model.device)
        with torch.no_grad():
            out = exp_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):].strip()
        
    for row in tqdm(val_rows, desc=f"Evaluating {label}"):
        narration = extract_narration_from_prompt(str(row.get("prompt", "")))
        gold = str(row.get("response", "")).strip()
        pred = exp_predict(narration)
        cs = char_similarity(pred, gold)
        tj = jaccard_token_similarity(pred, gold)
        exp_eval_rows.append(EvalRow(
            id=str(row.get("id", "")),
            narration=narration,
            gold=gold,
            pred=pred,
            exact_match=int(pred == gold),
            normalized_exact_match=int(normalize_text(pred) == normalize_text(gold)),
            char_similarity=cs,
            token_jaccard=tj,
            mse_char=(1.0 - cs) ** 2,
            mse_jaccard=(1.0 - tj) ** 2,
            txn_type=str(row.get("type", "unknown")),
        ))
        
    n = len(exp_eval_rows)
    exp_metrics = {
        "epsilon": epsilon,
        "label": label,
        "exact_match": sum(r.exact_match for r in exp_eval_rows) / max(1, n),
        "normalized_exact_match": sum(r.normalized_exact_match for r in exp_eval_rows) / max(1, n),
        "avg_char_similarity": mean(r.char_similarity for r in exp_eval_rows) if n else 0.0,
        "avg_token_jaccard": mean(r.token_jaccard for r in exp_eval_rows) if n else 0.0,
    }
    
    return exp_metrics

# Run experiments for the DP configurations
if USE_DP:
    for exp in EXPERIMENTS:
        if exp["epsilon"] != float("inf"):
            # Avoid repeating the target epsilon if it was already run in Section 16
            if exp["epsilon"] == TARGET_EPSILON and "dp_metrics" in globals():
                res = dict(dp_metrics)
                res["label"] = exp["label"]
                res["epsilon"] = exp["epsilon"]
                results.append(res)
            else:
                res = run_dp_experiment(exp["epsilon"], exp["label"])
                results.append(res)
                
    print("\nExperiment Results:")
    for r in results:
        print(f"{r['label']}: NEM={r['normalized_exact_match']:.3f}, CharSim={r['avg_char_similarity']:.3f}")
        
    plot_dp_privacy_utility_tradeoff(results, PLOTS_DP_DIR, show=SHOW_PLOTS)


## 19. Single narration demo


In [ ]:
sample_narration = "UPI/DR/867530921456/Reliance Fresh/Groceries/REF-2345/ICICI Bank"
print("Predicted payee:", predict_payee(sample_narration))


## 20. Next steps

- Increase `NUM_EPOCHS` if eval loss still drops.
- Try `Qwen/Qwen2.5-1.5B-Instruct` (more VRAM).
- Set `CALLBACK_EVAL_SAMPLES = 0` for faster training (fewer mid-run generations).
- Download `outputs/plots/training/`, `outputs/plots/evaluation/`, and `outputs/plots/dp/` for your thesis/report.
